<a href="https://colab.research.google.com/github/ricvazquez/ColabFiles/blob/main/Sesion10_Data_Profiling_Entregable_273509.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:**

**Matrícula:**

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [ ]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [ ]:
# Tu código aquí

## Primero creamos la copia
df_marketing_renombrado = df_marketing.copy()
print(df_marketing_renombrado.columns)
# # Aplicamos lower para unificar mayusculas y minisculas
#y como el objetivo es normalizar y en lo personal no me gusta el snakecase, tambine le voy a quitar los '_'
df_marketing_renombrado.columns =  df_marketing_renombrado.columns.str.lower().str.replace('_','')
#Y renombramos las celdas que pudieran no haber quedado al cine, asignandoles un unevo nombre
# Aunque no tiene un valor claro, cambiaremos en las variables mnt por mountain que supongo pudier aser lo mas aproximado
df_marketing_renombrado = df_marketing_renombrado.rename(columns={
    'mntwines': 'mountainwine',
    'mntfruits':'mountainfruits',
    'mntmeatproducts':'mountainproducts'})
df_marketing_renombrado.columns

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')


Index(['id', 'yearbirth', 'education', 'maritalstatus', 'income', 'kidhome',
       'teenhome', 'dtcustomer', 'recency', 'mountainwine', 'mountainfruits',
       'mountainproducts', 'mntfishproducts', 'mntsweetproducts',
       'mntgoldprods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'zcostcontact', 'zrevenue', 'response'],
      dtype='object')

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [ ]:
# Tu código aquí
# Para no repetir muchas veces el nombre de la columna, la guardamos en una variable
col = 'date_added'
# df_netflix.columns
# Imprimimos el antes para poder ve la diferencia
print('Antes:',df_netflix[col].isnull().sum())
print('Antes',df_netflix[col].dtypes)
#Aplicamos el datetime a la columna
df_netflix[col] = pd.to_datetime(df_netflix[col], format='mixed')
#E imprimos el despues para poder ver la diferencia
print('Despues:',df_netflix[col].dtypes)
print('Despues:',df_netflix[col].isnull().sum())
#Como podemos ver, los datos nulos no cambiaron, pero si cambio el type de la columna

Antes: 10
Antes object
Despues: datetime64[ns]
Despues: 10


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [ ]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [ ]:
# Tu código aquí
# Primero sacamos las filas con duplicados, para poder contrastar con el resultado
print('Filas con duplicados', len(df_marketing_dup))
#Sacamos los duplicados
print('Duplicados:',df_marketing_dup.duplicated().sum())
print('Duplicados por ID:',df_marketing_dup.duplicated(subset='ID').sum())
#Eliminamos los duplicados
df_marketing_dup = df_marketing_dup.drop_duplicates()
# Mostramos las filas y los duplicados final
print('Filas sin duplicados:', len(df_marketing_dup))
print('Duplicados final:',df_marketing_dup.duplicated().sum())

Filas con duplicados 2242
Duplicados: 2
Duplicados por ID: 2
Filas sin duplicados: 2240
Duplicados final: 0


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [ ]:
# Tu código aquí
#Primero vemos cuantos faltan por columna
print('Columnas con valores nulos', df_netflix.isnull().sum())
#Ahora vemos cuantos tienen solo 1 valor faltante por Fila
print('Filas con al menos un valor faltante:',df_netflix.isnull().any(axis=1).sum())

Columnas con valores nulos show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64
Filas con al menos un valor faltante: 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [ ]:
# Tu código aquí
# Primero guardamos en una variable los nulos y el total de las filas
nulos = df_netflix.isnull().sum()
total_filas = len(df_netflix)
#Guardamos el resultado en otra variable, aplicando la formula que nos dan al principio
comp = (1 - nulos / total_filas) * 100
# Imprmimos todas las columnas con su porcentaje
print('Completitud por columna:',comp)
# Y ahora escribimos la completitud mas baja, que podemos ver que es
# La completitud de Director
print('Completitud mas baja por Columna', comp.min())

Completitud por columna: show_id         100.000000
type            100.000000
title           100.000000
director         69.320663
cast             90.779504
country          93.489149
date_added       99.871581
release_year    100.000000
rating           99.910107
duration        100.000000
listed_in       100.000000
description     100.000000
dtype: float64
Completitud mas baja por Columna 69.32066264286631


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [ ]:
# Tu código aquí
# primero imprimimos lo que nos dice la practica para ver los diferentes tipos de valores que tiene el dataset en dicha columna
print(df_marketing['Marital_Status'].value_counts())
# Ahora retransformamos los datos, a algo que pueda considerar que tenga logica
# Lo mas sencillo seria hacerlo al azar, pero vamos a aprovechar que el csv cuenta con mas datos, como la fecha de nacimiento, la educacion ylos ingresos
df_revisar = df_marketing_dup[df_marketing_dup['Marital_Status'].isin(['Alone', 'Absurd', 'YOLO'])]
print(df_revisar[['Year_Birth', 'Education', 'Marital_Status','Income']])
# Viendo que aparentemente no hay in patron claro con el qu podamos decidir
# podriamos decir que los alone entrarian en la categoria de solteros
# Absurd son dos personas arriba de 30, con un buen salario, a las que posiblemente no les interese casarse, pero pudieran estar juntadas
# Yolo curiosamente son dos personas de la misma edad con el mismo ingreso el cual a simple vista hasta pudiera parecer un dato repetido
# Al ser ya personas de mas de 50 años, pudiera sonar a que son personas divorciadas
df_marketing['Marital_Status'] = df_marketing['Marital_Status'].replace({'Alone': 'Single', 'Absurd': 'Together', 'YOLO': 'Divorced'})
# Despues de hacer los nuevos cambios revisamos los valores
print(df_marketing['Marital_Status'].value_counts())

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64
      Year_Birth   Education Marital_Status   Income
131         1958      Master          Alone  61331.0
138         1973         PhD          Alone  35860.0
153         1988  Graduation          Alone  34176.0
2093        1993  Graduation         Absurd  79244.0
2134        1957      Master         Absurd  65487.0
2177        1973         PhD           YOLO  48432.0
2202        1973         PhD           YOLO  48432.0
Marital_Status
Married     864
Together    582
Single      483
Divorced    234
Widow        77
Name: count, dtype: int64


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [ ]:
# Tu código aquí
# Primero vamos a ver que imprime la columna que nos piden
print(df_netflix['show_id'])
# Asignamos el regex para hacer la busqyeda de cuales cumplen con el patron sn como se pide en la practica
patron = df_netflix['show_id'].str.match(r'^s\d+$')
# Mostramos el tamaño de los que si cumplen
print('Datos que cumplen:',len(df_netflix[patron]))
# Sacamos el porcentaje de cumplimento
print('Promedio',(len(df_netflix[patron])/len(df_netflix))*100,'%')
# print(patron)


0          s1
1          s2
2          s3
3          s4
4          s5
        ...  
7782    s7783
7783    s7784
7784    s7785
7785    s7786
7786    s7787
Name: show_id, Length: 7787, dtype: object
Datos que cumplen: 7787
Promedio 100.0 %


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [ ]:
# Tu código aquí
# Primero corremos el describe para ver los valores de la columna
print(df_marketing[['Year_Birth']].describe())

# Considerando que el año minimo es de 1893, y estamos en 2026, se antoja complicado que pudiera ser un dato real
# Pero seria bueno ver los demas datos minimos para asegurarnos, asi que reacomodamos la columna con orden ascendente de valores
min = df_marketing['Year_Birth'].sort_values(ascending=True)
# Como podemos ver los 3 valores mas bajos que son 1893, 1899, y 1900 son
#Seguramente errores de captura, teniendo un poco mas de validez los que siguen de 1940 en adelante
print(' Valores mas bajos',min.info)



        Year_Birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000
 Valores mas bajos <bound method Series.info of 239     1893
339     1899
192     1900
1950    1940
424     1941
        ... 
2213    1995
1850    1995
995     1995
1170    1996
46      1996
Name: Year_Birth, Length: 2240, dtype: int64>


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [ ]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [ ]:
# Paso 1 — ajuste de tipos
#Revisamos los types de cada columna
print(df_practica.dtypes)
# Como podemos ver, Income ahora es un tipo object, cuando deberia de ser un tipo float
# Asi que la convertimos
df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce')
print(df_practica.dtypes)
#despues de la conversion revisamos si todos son tipo numero

ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object
ID                       int64
Year_Birth               int64
Education               object
Marital_Status         

In [ ]:
# Paso 2 — duplicados
# Primero mostramos las filas duplicadas por columnas
print('Duplicados:',df_practica.duplicated().sum())
# print('Duplicados por ID:',df_practica.duplicated(subset='ID').sum())
#Eliminamos los duplicados
df_practica = df_practica.drop_duplicates()
# Mostramos las filas y los duplicados final
print('Duplicados:',df_practica.duplicated().sum())

Duplicados: 1
Duplicados: 0


In [ ]:
# Paso 3 — valores faltantes
# Ahora revisamos cuales son los valores nulo o faltantes
print('Columnas con valores nulos', df_practica.isnull().sum())

# Ahora que sabemos que el valor nulo esta en la columna income, podemos ver cual es
print('Valor nulo en income',df_practica[df_practica['Income'].isnull()])
# Coincide con el valor que se agrego arriba como texto el cual al momento de hacerlo
#tipo numerico, se convirtio a NaN, en la fila 5

Columnas con valores nulos ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64
Valor nulo en income      ID  Year_Birth   Education Marital_Status  Income  Kidhome  Teenhome  \
5  5314        1951  Graduation       Together     NaN        0         1   

  Dt_Customer  Recency  MntWines  ...  NumWebVisitsMonth  A

In [ ]:
# Paso 4 — exploración categórica
# Ahora vamos a ver que valores tiene marital status
print(df_practica['Marital_Status'].unique())
# hay que recordar que marital status ya la normalizamos en la actividad 6, y todos los demas campos
# Parecen estar escritos igual, entonces no creo sea necesario normalizarlo
# Lo que pudieramos hacer, es pasarlo de object a category, para que ocupe menos recursos, aprovechando que son solo 4 opciones difernetes
df_practica['Marital_Status'] = df_practica['Marital_Status'].astype('category')
print(df_practica['Marital_Status'].unique())


['Together' 'Single' 'Married' 'Divorced']
['Together', 'Single', 'Married', 'Divorced']
Categories (4, object): ['Divorced', 'Married', 'Single', 'Together']


**Tu reporte de profiling:**

*(Escribe aquí tu resumen de 3-4 líneas)*

**Hallazgos**

Despues del trabajo hecho con el dataset practica, podemos ver como cambiar el formato a las columnas para darle un mejor manejo y poder segurarse que todas las filas correspondan al tipo de dato que queremos utilizar, tambien a eliminar duplicados, aunque solo haya sido uno, en el tercer punto pudimos ver como el dato que se le inyecto artificialmente paso a ser nulo con la transofromacion de la columna a float. Y en el ultimo paso definimos si era necesario normalizar la columna o no, al notar que los datos unicos se repetian, pudimos ver que la columna ya estaba normalizada, lo que si hice fue categorizarla, al no tener mucha variedad en su contenido, lo recomendable es categorizarla para que en un momento dado que necesitamos trabajar con ella, consuma menos recursos.

En conclusion

Podemos entender el porque es necesario hacer un perfilado de datos el cual no es mas que un proceso donde examinamos, analizamos y resumimos los datos. Esto nos ayuda a entender la estructura, el contenido y se pudiera decir que la co relacion entre las columnas.